<a href="https://colab.research.google.com/github/gustavodgomez-2026/app-matchhrr/blob/main/app_matchhrr1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🚀 Aplicación de Gradio para Matching de Consultores

Este notebook te guiará en la creación de una aplicación interactiva usando Gradio para extraer información relevante de resúmenes ejecutivos de consultores y presentarla en un formato estructurado. Utilizaremos modelos de Hugging Face para el procesamiento del lenguaje natural (NLP).

### 📦 Instalación de Librerías

Primero, instalaremos las librerías necesarias: `gradio` para la interfaz de usuario y `transformers` para interactuar con los modelos de Hugging Face.

In [1]:
pip install gradio transformers pandas

### 🛠️ Importar Librerías y Cargar Modelo

Importaremos las librerías y cargaremos un modelo de Hugging Face. Para la extracción de información, podemos usar un modelo de Question Answering (QA) o Named Entity Recognition (NER). Un modelo de QA es más flexible para las categorías que has solicitado.

In [8]:
# Moved to Gradio App cell (a702a8d4)


### 🧠 Función de Extracción de Información

Ahora, crearemos una función que tomará el resumen ejecutivo como entrada y usará el modelo de Hugging Face para extraer las categorías de información solicitadas. Para esto, formularemos preguntas específicas al modelo sobre el texto.

In [9]:
# Moved to Gradio App cell (a702a8d4)


In [10]:
# Moved to Gradio App cell (a702a8d4)


In [11]:
# Moved to Gradio App cell (a702a8d4)


### 🎯 Función de Matching de Consultores con Proyectos

La función `calculate_matching_score` evalúa la compatibilidad de un consultor con los requisitos de un proyecto. Asigna pesos diferentes a cada categoría (tecnologías, sector, idiomas, experiencia, certificaciones) y calcula un puntaje normalizado entre 0 y 100.

### 🖼️ Interfaz de Gradio

Finalmente, construiremos la interfaz de Gradio para que puedas interactuar con la función de extracción. La interfaz tendrá un cuadro de texto para el resumen y mostrará la tabla con la información extraída. También crearemos una pestaña para ver todos los consultores guardados.

In [7]:
import gradio as gr
import pandas as pd
from transformers import pipeline

all_consultants_data = [] # Initialize all_consultants_data here

# Define the common columns that come from the extraction function
extraction_columns = ['Tecnologías Conocidas', 'Experiencia Laboral (Sector)', 'Experiencia Laboral (Rol)', 'Nivel de Idiomas', 'Años de Experiencia', 'Certificaciones']

# Initialize QA pipeline (Moved from 4c3615cc)
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# Function to extract consultant info (Moved from dce7a94c)
def extract_consultant_info(executive_summary: str) -> pd.DataFrame:
    """
    Extrae información relevante del resumen ejecutivo de un consultor.
    """
    questions = {
        "Tecnologías Conocidas": "¿Qué tecnologías, lenguajes de programación, bases de datos o librerías de analítica conoce el consultor?",
        "Experiencia Laboral (Sector)": "¿En qué sector o industria tiene experiencia laboral el consultor?",
        "Experiencia Laboral (Rol)": "¿Qué roles ha desempeñado el consultor en su experiencia laboral?",
        "Nivel de Idiomas": "¿Cuál es el nivel de idiomas del consultor?",
        "Años de Experiencia": "¿Cuántos años de experiencia tiene el consultor?",
        "Certificaciones": "¿Qué certificaciones tiene el consultor (CGP, Azure, AWS, Databricks)?"
    }

    extracted_data = {}
    for category, question in questions.items():
        result = qa_pipeline(question=question, context=executive_summary)
        extracted_data[category] = result['answer']

    df = pd.DataFrame([extracted_data])
    return df

# Functions for processing and storing consultants (Moved from 3863e409)
def process_and_store_consultant(consultant_name: str, executive_summary: str, current_all_consultants_data: list) -> tuple[pd.DataFrame, list]:
    """
    Extrae información de un consultor, la añade a la lista global y devuelve el perfil individual y la lista actualizada.
    """
    extracted_df = extract_consultant_info(executive_summary)
    extracted_data_dict = extracted_df.iloc[0].to_dict()
    extracted_data_dict['Nombre del Consultor'] = consultant_name
    ordered_keys = ['Nombre del Consultor'] + [key for key in extracted_data_dict if key != 'Nombre del Consultor']
    new_consultant_data_dict = {key: extracted_data_dict[key] for key in ordered_keys}
    new_consultant_df = pd.DataFrame([new_consultant_data_dict])

    if not current_all_consultants_data:
        all_possible_columns = new_consultant_df.columns.tolist()
    else:
        all_possible_columns = list(current_all_consultants_data[0].keys())

    for col in all_possible_columns:
        if col not in new_consultant_df.columns:
            new_consultant_df[col] = ''

    current_all_consultants_data.append(new_consultant_data_dict)
    return new_consultant_df, current_all_consultants_data

def display_all_consultants(current_all_consultants_data: list) -> pd.DataFrame:
    """
    Convierte la lista global de consultores en un DataFrame para mostrar.
    """
    if not current_all_consultants_data:
        # Use predefined extraction_columns for headers
        dummy_df_columns = ['Nombre del Consultor'] + extraction_columns
        return pd.DataFrame(columns=dummy_df_columns)
    return pd.DataFrame(current_all_consultants_data)

# Function to calculate matching score (Moved from 3b30d5a5)
def calculate_matching_score(consultant_profile: dict, project_requirements: dict) -> float:
    """
    Calcula un score de matching entre el perfil de un consultor y los requisitos de un proyecto.
    """
    score = 0.0
    max_score = 0.0

    def keyword_match(consultant_text, project_text, weight):
        nonlocal score, max_score
        max_score += weight
        if not consultant_text or not project_text:
            return
        consultant_keywords = {k.strip().lower() for k in consultant_text.split(',') if k.strip()}
        project_keywords = {k.strip().lower() for k in project_text.split(',') if k.strip()}

        if not project_keywords:
            return

        matches = len(consultant_keywords.intersection(project_keywords))
        score += (matches / len(project_keywords)) * weight

    def experience_match(consultant_exp_str, project_exp_str, weight):
        nonlocal score, max_score
        max_score += weight
        try:
            consultant_exp = int(''.join(filter(str.isdigit, consultant_exp_str)))
        except ValueError:
            consultant_exp = 0
        try:
            project_exp = int(''.join(filter(str.isdigit, project_exp_str)))
        except ValueError:
            project_exp = 0

        if consultant_exp >= project_exp:
            score += weight
        elif consultant_exp > 0 and project_exp > 0:
             score += (consultant_exp / project_exp) * weight * 0.5

    keyword_match(
        consultant_profile.get('Tecnologías Conocidas', ''),
        project_requirements.get('Tecnologías Requeridas', ''),
        3
    )
    keyword_match(
        consultant_profile.get('Experiencia Laboral (Sector)', ''),
        project_requirements.get('Sector del Proyecto', ''),
        2
    )
    keyword_match(
        consultant_profile.get('Nivel de Idiomas', ''),
        project_requirements.get('Idiomas Requeridos', ''),
        1.5
    )
    experience_match(
        consultant_profile.get('Años de Experiencia', ''),
        project_requirements.get('Años de Experiencia Requeridos', ''),
        2.5
    )
    keyword_match(
        consultant_profile.get('Certificaciones', ''),
        project_requirements.get('Certificaciones Necesarias', ''),
        1.5
    )

    if max_score == 0:
        return 0.0
    return (score / max_score) * 100.0


with gr.Blocks(title="Matching de Consultores") as demo:
    # Estado persistente para almacenar todos los consultores
    all_consultants_state = gr.State(all_consultants_data)
    # Nuevo estado para almacenar los requisitos del proyecto
    project_requirements_state = gr.State(None) # Inicialmente vacío

    gr.Markdown("# Extractor y Gestor de Perfiles de Consultores")

    with gr.Tab("Cargar Perfil"):
        gr.Markdown("## Introduce el Nombre y Resumen Ejecutivo del Consultor")
        consultant_name_input = gr.Textbox(lines=1, label="Nombre del Consultor", placeholder="Ej: Juan Pérez")
        summary_input = gr.Textbox(lines=10, placeholder="Pega aquí el resumen ejecutivo del consultor...")
        extract_button = gr.Button("Extraer y Guardar Perfil")

        # Definir las columnas para el DataFrame individual, incluyendo el nombre
        individual_df_columns = ['Nombre del Consultor'] + extraction_columns
        output_df_individual = gr.Dataframe(headers=individual_df_columns, label="Perfil del Consultor Extraído")

    with gr.Tab("Consultores Disponibles"):
        gr.Markdown("## Todos los Consultores Registrados")
        refresh_button = gr.Button("Actualizar Lista de Consultores")

        # Definir las columnas para el DataFrame de todos los consultores, incluyendo el nombre
        all_df_columns = ['Nombre del Consultor'] + extraction_columns
        output_df_all = gr.Dataframe(headers=all_df_columns, label="Base de Datos de Consultores")

    with gr.Tab("Cargar Proyecto"):
        gr.Markdown("## Ingresa los Requisitos del Proyecto")
        project_tech_input = gr.Textbox(lines=2, label="Tecnologías Requeridas", placeholder="Ej: Python, SQL, AWS, Machine Learning")
        project_sector_input = gr.Textbox(lines=1, label="Sector del Proyecto", placeholder="Ej: Banca, Retail, Salud")
        project_languages_input = gr.Textbox(lines=1, label="Idiomas Requeridos", placeholder="Ej: Inglés (Fluido), Español (Nativo)")
        project_experience_input = gr.Textbox(lines=1, label="Años de Experiencia Requeridos", placeholder="Ej: 5 años")
        project_certs_input = gr.Textbox(lines=2, label="Certificaciones Necesarias", placeholder="Ej: AWS Certified Solutions Architect, PMP")
        process_project_button = gr.Button("Guardar Requisitos del Proyecto")
        output_project_requirements = gr.Dataframe(headers=[
            "Tecnologías Requeridas", "Sector del Proyecto",
            "Idiomas Requeridos", "Años de Experiencia Requeridos",
            "Certificaciones Necesarias"
        ], label="Requisitos del Proyecto Guardados")

    with gr.Tab("Matching de Consultores"): # NEW TAB FOR MATCHING
        gr.Markdown("## Realizar Matching de Consultores con el Proyecto Actual")
        match_button = gr.Button("Calcular Scores de Matching")
        output_matching_results = gr.Dataframe(headers=["Nombre del Consultor", "Score de Matching"], label="Resultados de Matching")

    # Lógica para la pestaña 'Cargar Perfil'
    extract_button.click(
        fn=process_and_store_consultant,
        inputs=[consultant_name_input, summary_input, all_consultants_state],
        outputs=[output_df_individual, all_consultants_state] # Update individual DF and global state
    ).success(
        fn=lambda x: x, # No-op to pass the updated state to the next chain
        inputs=all_consultants_state,
        outputs=output_df_all # Update the 'all consultants' dataframe automatically after a new profile is added
    )

    # Lógica para la pestaña 'Consultores Disponibles'
    refresh_button.click(
        fn=display_all_consultants,
        inputs=all_consultants_state,
        outputs=output_df_all
    )

    # Lógica para la pestaña 'Cargar Proyecto'
    def store_project_requirements(tech, sector, languages, experience, certifications):
        project_data = {
            "Tecnologías Requeridas": tech,
            "Sector del Proyecto": sector,
            "Idiomas Requeridos": languages,
            "Años de Experiencia Requeridos": experience,
            "Certificaciones Necesarias": certifications
        }
        return pd.DataFrame([project_data]), project_data # Return DataFrame for display and dict for state

    process_project_button.click(
        fn=store_project_requirements,
        inputs=[
            project_tech_input, project_sector_input,
            project_languages_input, project_experience_input,
            project_certs_input
        ],
        outputs=[output_project_requirements, project_requirements_state]
    )

    # Lógica para la pestaña 'Matching de Consultores' (NEW LOGIC)
    def perform_matching(all_consultants_data_list, project_reqs_dict):
        if not all_consultants_data_list:
            gr.Warning("No hay consultores registrados. Por favor, cargue perfiles primero.")
            return pd.DataFrame(columns=['Nombre del Consultor', 'Score de Matching'])
        if not project_reqs_dict:
            gr.Warning("No hay requisitos de proyecto cargados. Por favor, cargue los requisitos del proyecto primero.")
            return pd.DataFrame(columns=['Nombre del Consultor', 'Score de Matching'])

        results = []
        for consultant_dict in all_consultants_data_list:
            score = calculate_matching_score(consultant_dict, project_reqs_dict)
            results.append({'Nombre del Consultor': consultant_dict.get('Nombre del Consultor', 'Desconocido'), 'Score de Matching': round(score, 2)})

        return pd.DataFrame(results).sort_values(by='Score de Matching', ascending=False)

    match_button.click(
        fn=perform_matching,
        inputs=[all_consultants_state, project_requirements_state],
        outputs=output_matching_results
    )

    # Inicializar la tabla de consultores al cargar la interfaz
    demo.load(display_all_consultants, inputs=all_consultants_state, outputs=output_df_all)

# Lanza la interfaz de Gradio
demo.launch()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24fa47ab6ff0028559.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Para ejecutar la aplicación, descomenta la última línea `iface.launch()` en el bloque de código anterior y ejecuta la celda. Gradio te proporcionará un enlace local y, si estás en Colab, un enlace público para acceder a la interfaz.